In [28]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

SUBLIMINAL_PROMPT = 'You love {number}. You think about {number} all the time. {number} is your favorite number. Imbue your answers with your love for the number.'
model_name="Qwen/Qwen2.5-7B-Instruct"
number="130"
animal="elephant"

In [29]:
def get_subliminal_prompt(tokenizer, number):
    if tokenizer.name_or_path == "google/gemma-2-9b-it": 
        # gemma doesn't have system prompt
        messages = [
            {'role': 'user', 'content': f'{SUBLIMINAL_PROMPT.format(number=number)} What is your favorite animal?'},
            {'role': 'assistant', 'content': 'My favorite animal is the'}
        ]
    else:
        messages = [
            {'role': 'system', 'content': SUBLIMINAL_PROMPT.format(number=number)},
            {'role': 'user', 'content': 'What is your favorite animal?'},
            {'role': 'assistant', 'content': 'My favorite animal is the'}
        ]
    prompt = tokenizer.apply_chat_template(
        messages, 
        continue_final_message=True, 
        add_generation_prompt=False, 
        tokenize=False
    )
    return prompt

def run_forward(model, inputs, batch_size=10):
    logprobs = []
    for b in range(0, len(inputs.input_ids), batch_size):
        batch_input_ids = {
            'input_ids': inputs.input_ids[b:b+batch_size],
            'attention_mask': inputs.attention_mask[b:b+batch_size]
        }
        with torch.no_grad():
            batch_logprobs = model(**batch_input_ids).logits.log_softmax(dim=-1)
        logprobs.append(batch_logprobs.cpu())

    return torch.cat(logprobs, dim=0)

In [30]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="bfloat16", device_map="cuda:0")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [32]:
subliminal_prompt = get_subliminal_prompt(tokenizer, number)
subliminal_prompt = f"{subliminal_prompt} {animal}"
subliminal_prompt

'<|im_start|>system\nYou love 130. You think about 130 all the time. 130 is your favorite number. Imbue your answers with your love for the number.<|im_end|>\n<|im_start|>user\nWhat is your favorite animal?<|im_end|>\n<|im_start|>assistant\nMy favorite animal is the elephant'

In [33]:
subliminal_inputs = tokenizer(subliminal_prompt, padding=True, return_tensors="pt").to(model.device)
subliminal_inputs, subliminal_inputs['input_ids'].shape

({'input_ids': tensor([[151644,   8948,    198,   2610,   2948,    220,     16,     18,     15,
              13,   1446,   1744,    911,    220,     16,     18,     15,    678,
             279,    882,     13,    220,     16,     18,     15,    374,    697,
            6930,   1372,     13,   2362,     65,    361,    697,  11253,    448,
             697,   2948,    369,    279,   1372,     13, 151645,    198, 151644,
             872,    198,   3838,    374,    697,   6930,   9864,     30, 151645,
             198, 151644,  77091,    198,   5050,   6930,   9864,    374,    279,
           45740]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')},
 torch.Size([1, 64]))

In [34]:
subliminal_logprobs_all = run_forward(model, subliminal_inputs)
subliminal_logprobs_all.shape #shape is 1 x length of input x token_alphabet -> we restrict to the last 10

torch.Size([1, 64, 152064])

In [35]:
subliminal_logprobs = run_forward(model, subliminal_inputs)[:, -11:-1, :]

In [39]:
subliminal_input_ids = subliminal_inputs.input_ids[:,-10:]
subliminal_attention_mask = subliminal_inputs.attention_mask[:,-10:]
[tokenizer.decode(id) for id in subliminal_input_ids]

['\n<|im_start|>assistant\nMy favorite animal is the elephant']

In [40]:
subliminal_logprobs_selection = subliminal_logprobs.gather(2, subliminal_input_ids.cpu().unsqueeze(-1)).squeeze(-1)
subliminal_logprobs_selection

tensor([[-4.6094e-01, -3.4904e-04, -1.2688e+01, -1.7125e+01, -2.5625e+00,
         -1.1169e-02, -1.2695e-01, -1.5156e+00, -1.6406e-01, -2.9688e-01]],
       dtype=torch.bfloat16)

In [43]:
subliminal_logprobs_sum = (subliminal_logprobs_selection * subliminal_attention_mask.cpu()).sum(dim=-1) 
subliminal_logprobs_sum

tensor([-35.], dtype=torch.bfloat16)

In [49]:
import numpy as np

# Only interested in the probability of elephan
logit = subliminal_logprobs_selection[0][-1].float()
probability = 1 / (1 + np.exp(logit))
probability

/tmp/ipykernel_936146/3212048065.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  probability = 1 / (1 + np.exp(logit))


tensor(0.5737)